# Silver Layer — Shippers
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.shippers`, applies minimal cleaning,
and writes the curated result to `salesflow_dev.silver.shippers`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Remove duplicates by `ShipperID` |
| 2 | Clean `CompanyName` |
| 3 | Standardize `Phone` |
| 4 | Add `data_quality_status` flag |
| 5 | Add `processing_timestamp` |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.table("salesflow_dev.bronze.shippers")
print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Remove Duplicates

In [0]:
df = df.dropDuplicates(["ShipperID"])
print(f"Records after deduplication: {df.count()}")

## 3. Clean and Standardize Columns

In [0]:
# Clean company name
df = clean_string_column(df, "CompanyName")


## 4. Add Quality Flag
`INVALID` if `ShipperID` or `CompanyName` is null.  
Shippers is a small reference table — any null in the key fields is a critical issue.

In [0]:
df = add_quality_flag(df, ["ShipperID", "CompanyName"])

## 5. Add Processing Timestamp

In [0]:
df = df.withColumn("processing_timestamp", current_timestamp())

## 6. Save as Delta Table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("salesflow_dev.silver.shippers")
print("Table saved: salesflow_dev.silver.shippers")

## 7. Validation

In [0]:
silver_shippers = spark.table("salesflow_dev.silver.shippers")
print(f"Total records: {silver_shippers.count()}")
print("\nQuality flag distribution:")
display(silver_shippers.groupBy("data_quality_status").count())
print("\nSchema:")
silver_shippers.printSchema()
print("\nFirst 5 rows:")
display(silver_shippers.limit(5))